# Modelo OLS — preparación consolidada para Feature Selection

Este notebook recupera evidencias históricas y prepara `df_trabajo`. Su ejecución no entrena
modelos ni ejecuta clustering, selección o regularización. Los procedimientos reproducibles
se centralizan en `experimentos_ols.py` y `transformaciones_experimentales.py` y solo se ejecutan
cuando se invoquen explícitamente en una etapa posterior. La combinación seleccionada es provisional.


## Dataset base y particiones persistidas
Una sola lectura de `PropUsadas_OLS.csv`. Las proporciones nominales son 70/10/20; se conservan
los cortes por meses completos y sus tamaños efectivos. Test se conserva como filas reservadas,
sin ajustar transformadores ni evaluar precios con él.


In [3]:
from pathlib import Path
import hashlib
import sys
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display

raiz = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "Ventas" / "Propiedades_Usadas").is_dir()
             and (p / "requirements.txt").is_file()), None)
assert raiz is not None, "Ejecutar dentro del proyecto Moscow_EDA."
assert Path(sys.prefix).resolve() == (raiz / ".venv").resolve(), "Seleccionar el kernel de .venv."
archivo_datos = raiz / "Ventas" / "Propiedades_Usadas" / "PropUsadas_OLS.csv"
datos_base = pd.read_csv(archivo_datos, parse_dates=["date_posted_dt"])
datos = datos_base  # Alias de lectura; las variantes usan copias.
assert not datos.empty and datos.columns.is_unique
assert datos["id"].notna().all() and datos["id"].is_unique
assert datos["date_posted_dt"].notna().all() and datos["date_posted_dt"].is_monotonic_increasing
assert datos["price_rub"].notna().all() and datos["price_rub"].gt(0).all()
assert np.isfinite(datos["price_rub"]).all()
orden_splits = ["train", "validation", "test"]
assert datos["dataset_split"].notna().all() and set(datos["dataset_split"]) == set(orden_splits)
assert datos.groupby("date_posted_dt")["dataset_split"].nunique().eq(1).all()
assert datos.groupby("year_month")["dataset_split"].nunique().eq(1).all()
assert datos["year_month"].eq(datos["date_posted_dt"].dt.strftime("%Y-%m")).all()

conjuntos = {nombre: datos.loc[datos["dataset_split"].eq(nombre)].copy() for nombre in orden_splits}
resumen_particiones = datos.groupby("dataset_split").agg(
    observaciones=("id", "size"), desde=("date_posted_dt", "min"), hasta=("date_posted_dt", "max")
).reindex(orden_splits)
resumen_particiones["columnas"] = datos.shape[1]
resumen_particiones["porcentaje"] = resumen_particiones["observaciones"] / len(datos) * 100
for anterior, siguiente in zip(orden_splits, orden_splits[1:]):
    assert resumen_particiones.loc[anterior, "hasta"] < resumen_particiones.loc[siguiente, "desde"]
assert sum(len(d) for d in conjuntos.values()) == len(datos)
display(resumen_particiones)


,observaciones,desde,hasta,columnas,porcentaje
dataset_split,,,,,
train,34853,2020-01-01,2024-05-31,38,70.052057
validation,5258,2024-06-01,2025-01-31,38,10.568207
test,9642,2025-02-01,2026-04-29,38,19.379736


## Diccionario y características originales
El catálogo conserva el baseline original y distingue predictoras, objetivo y auxiliares.
Las características de trabajo se definen por separado; no se altera este baseline.


In [5]:
# nombre, descripción, rol, tratamiento
especificacion = [
    ("id", "Identificador único del anuncio; solo trazabilidad.", "auxiliar", "excluir"),
    ("price_rub", "Precio publicado en rublos, sin logaritmo.", "objetivo", "objetivo"),
    ("lat", "Latitud de la propiedad en grados.", "predictora", "numerica"),
    ("lon", "Longitud de la propiedad en grados.", "predictora", "numerica"),
    ("total_area", "Superficie total en m².", "predictora", "numerica"),
    ("living_area", "Superficie habitable en m².", "predictora", "numerica"),
    ("kitchen_area", "Superficie de cocina en m².", "predictora", "numerica"),
    ("rooms", "Cantidad de habitaciones.", "predictora", "numerica"),
    ("floor", "Piso de la propiedad.", "predictora", "numerica"),
    ("total_floors", "Cantidad de pisos del edificio.", "predictora", "numerica"),
    ("building_year", "Año de construcción; nulo si era posterior al anuncio.", "predictora", "numerica"),
    ("ceiling_height", "Altura de techo en metros.", "predictora", "numerica"),
    ("metro_distance_min", "Tiempo de acceso al metro en minutos.", "predictora", "numerica"),
    ("to_center_km", "Distancia de la propiedad al centro en km.", "predictora", "numerica"),
    ("mortgage_rate_at_listing", "Tasa hipotecaria al publicar, en porcentaje.", "predictora", "numerica"),
    ("metro_lat", "Latitud de la estación de metro en grados.", "predictora", "numerica"),
    ("metro_lon", "Longitud de la estación de metro en grados.", "predictora", "numerica"),
    ("metro_year_opened", "Año de apertura de la estación.", "predictora", "numerica"),
    ("metro_to_center_km", "Distancia de la estación al centro en km.", "predictora", "numerica"),
    ("has_balcony", "Balcón: 0=no, 1=sí.", "predictora", "numerica"),
    ("metro_distance_type", "Modalidad de acceso: 0=a pie, 1=transporte.", "predictora", "numerica"),
    ("seller_type", "Vendedor: 0=propietario, 1=agencia.", "predictora", "numerica"),
    ("building_year_inconsistent", "1 si el año de construcción informado era posterior al anuncio.", "predictora", "numerica"),
    ("months_since_start", "Tendencia mensual con enero de 2020 igual a cero.", "predictora", "numerica"),
    ("n_listings_secondary_lag1", "Anuncios usados del distrito/okrug en el mes anterior.", "predictora", "numerica"),
    ("n_listings_newbuild_lag1", "Anuncios nuevos del distrito/okrug en el mes anterior.", "predictora", "numerica"),
    ("n_listings_rental_lag1", "Anuncios de alquiler del distrito/okrug en el mes anterior.", "predictora", "numerica"),
    ("cbr_key_rate_pct_lag1", "Tasa de referencia del BCR del mes anterior, en porcentaje.", "predictora", "numerica"),
    ("lag1_unavailable", "1 si falta al menos un indicador del mes anterior.", "predictora", "numerica"),
    ("district", "Distrito de Moscú.", "predictora", "onehot"),
    ("building_type", "Tipo de construcción del edificio.", "predictora", "onehot"),
    ("renovation", "Estado de renovación; sin escala ordinal impuesta.", "predictora", "onehot"),
    ("metro_line", "Línea de metro asociada al anuncio.", "predictora", "onehot"),
    ("month_of_year", "Mes calendario 1–12; se trata como categoría estacional.", "predictora", "onehot"),
    ("okrug", "División administrativa; redundante con district, solo auditoría.", "auxiliar", "excluir"),
    ("date_posted_dt", "Fecha principal para ordenar y construir folds temporales.", "auxiliar", "excluir"),
    ("year_month", "Período YYYY-MM para agrupar meses completos.", "auxiliar", "excluir"),
    ("dataset_split", "Asignación persistida: train, validation o test.", "auxiliar", "excluir"),
]
diccionario = pd.DataFrame(especificacion, columns=["nombre", "descripcion", "rol", "tratamiento"])
assert diccionario["nombre"].is_unique
assert set(diccionario["nombre"]) == set(datos), "El esquema del CSV cambió: revisar el catálogo."
diccionario["tipo"] = diccionario["nombre"].map(datos.dtypes.astype(str))
display(diccionario.groupby("rol").size().rename("variables").to_frame())

objetivo = diccionario.loc[diccionario["rol"].eq("objetivo"), "nombre"].item()
columnas_numericas = diccionario.loc[diccionario["tratamiento"].eq("numerica"), "nombre"].tolist()
columnas_ohe = diccionario.loc[diccionario["tratamiento"].eq("onehot"), "nombre"].tolist()
predictoras = columnas_numericas + columnas_ohe
auxiliares = diccionario.loc[diccionario["rol"].eq("auxiliar"), "nombre"].tolist()
prohibidas = {
    "price_per_sqm", "secondary_price_per_sqm", "newbuild_price_per_sqm",
    "rental_price_per_sqm_monthly", "secondary_mom_change_pct", "newbuild_mom_change_pct",
    "n_listings_secondary", "n_listings_newbuild", "n_listings_rental",
    "cbr_key_rate_pct", "avg_mortgage_rate_pct",
}
assert not set(predictoras).intersection(prohibidas | set(auxiliares) | {objetivo})
assert all(pd.api.types.is_numeric_dtype(datos[c]) for c in columnas_numericas)
predictoras_base = list(predictoras)


,variables
rol,
auxiliar,5
objetivo,1
predictora,32


## MLflow y DagsHub
Se usa el experimento existente `Ventas_Propiedades_Usadas` y credenciales del entorno.
Los Runs históricos se leen; la única escritura es un artefacto ligero de configuración provisional
en el Run K-Means K=8, sin alterar sus métricas, parámetros ni notas.


In [7]:
import os
import subprocess
from dotenv import load_dotenv

archivo_env = raiz / ".env"
if archivo_env.is_file():
    assert subprocess.run(["git", "check-ignore", "-q", ".env"], cwd=raiz, capture_output=True).returncode == 0
    assert subprocess.run(["git", "ls-files", "--error-unmatch", ".env"], cwd=raiz, capture_output=True).returncode != 0
    load_dotenv(archivo_env, override=False)
assert os.environ.get("DAGSHUB_USER_TOKEN", "").strip(), "Definir DAGSHUB_USER_TOKEN sin incluirlo en el notebook."

DAGSHUB_OWNER = "matiasgastonlonghino"
DAGSHUB_REPO = "MOSCOW_IDA"
NOMBRE_EXPERIMENTO = "Ventas_Propiedades_Usadas"
TRACKING_URI = f"https://dagshub.com/{DAGSHUB_OWNER}/{DAGSHUB_REPO}.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ.get("DAGSHUB_USERNAME") or DAGSHUB_OWNER
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ["DAGSHUB_USER_TOKEN"]
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
os.environ["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "30"
os.environ["MLFLOW_HTTP_REQUEST_MAX_RETRIES"] = "1"

import mlflow
from mlflow.tracking import MlflowClient

assert mlflow.active_run() is None, "Finalizar el Run activo antes de cambiar la configuración."
mlflow.set_tracking_uri(TRACKING_URI)
cliente_mlflow = MlflowClient(tracking_uri=TRACKING_URI)
_ = cliente_mlflow.search_experiments(max_results=1)
experimento_existente = cliente_mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)
if experimento_existente is None or experimento_existente.lifecycle_stage != "active":
    raise RuntimeError("Debe existir Ventas_Propiedades_Usadas; no se crean experimentos nuevos.")
print("Conexión autenticada de lectura verificada:", mlflow.get_tracking_uri())
print("Experimento:", NOMBRE_EXPERIMENTO,
      "(existente)")
print("Versiones:", {p: version(p) for p in ["pandas", "scikit-learn", "mlflow", "python-dotenv"]})


Conexión autenticada de lectura verificada: https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow
Experimento: Ventas_Propiedades_Usadas (existente)
Versiones: {'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'mlflow': '3.17.1', 'python-dotenv': '1.2.4'}


## Funciones reutilizables
Preprocesamiento, ocho métricas, OLS, CV temporal, agrupamientos y registro están centralizados.
Imputación, encoding, escalado opcional y Duan se ajustarán solo con el entrenamiento correspondiente.
R² ajustado en RUB es descriptivo; los AIC/BIC históricos usan likelihood lognormal en RUB con
Jacobiano. Los diagnósticos nativos de Train en log(RUB) se conservan en los Runs de los SET.


In [9]:
import json
from tempfile import TemporaryDirectory
from IPython.display import Markdown, HTML

carpeta_notebook = raiz / "Ventas" / "Propiedades_Usadas"
if str(carpeta_notebook) not in sys.path:
    sys.path.insert(0,str(carpeta_notebook))
from experimentos_ols import (
    crear_preprocesador, evaluar_regresion, tabla_metricas, ajustar_ols,
    validacion_temporal, estimar_efectos_distrito, agrupar_efectos,
    registrar_experimento, recuperar_runs, tabla_markdown,
)
from transformaciones_experimentales import (
    ARTEFACTO_MAPAS, huella_tabla, huella_json, contexto_entrenamiento,
    recuperar_agrupamiento, leer_json_remoto, preparar_df_seleccionado,
    preparar_fold_seleccionado,
)

ESCALAR = False
N_FOLDS, MESES_VALID_FOLD = 5,6
config_preprocesamiento = {
    "imputacion_numerica":"median", "imputacion_categorica":"most_frequent",
    "keep_empty_features":True, "ohe_drop":"first", "ohe_unknown":"ignore",
    "escalar":ESCALAR, "objetivo":objetivo, "escala_metricas":"RUB",
    "particion":"dataset_split_persistido",
}
huella_datos = hashlib.sha256(archivo_datos.read_bytes()).hexdigest()
huella_base_memoria = huella_tabla(datos_base)
cache_transformaciones = {}
RUNS_REFERENCIA = {'Baseline original': '03bf0a40212846eebf7eb00d68f3c634', 'Baseline log': 'ff680a2eb84d4dcb9d1bb64b2ac55036', 'KMeans K=3': '38d09624693e4e9eac36c14c60ccba12', 'KMeans K=4': '33c1f3a196c543ca8776a92db9523827', 'KMeans K=5': '7acb296b04c24338abd5af4bc12c7fa9', 'KMeans K=6': 'f1429a8e41144005ad0563a74f15d687', 'KMeans K=7': 'ef7ee441b0f941609ed81891eb6a2801', 'KMeans K=8': 'ab4ff01bcb544784bf2e33312e128c29', 'KMeans K=9': '86d6fdec3eb349d1a11ed7a930ccb2f8', 'KMeans K=10': '29934496f8e04aeda1819706460b82dd', 'Ward K=3': 'ebce827914b44a15aead93ed5798234a', 'Ward K=4': '51ef3dcdf4294391bbb0b220f4d2727d', 'Ward K=5': '81e87497a40f491eb537863518009aae', 'Ward K=6': 'e9a6012ae7a2451c93e8ef3a2ce01086', 'Ward K=7': '85c942d52cd745deaa617273cfecb2ae', 'Ward K=8': '3025eb1238a843e3b128a87db8835dfb', 'Ward K=9': '27548a8e778346789a16e9591014c086', 'Ward K=10': '4d66a2acee91452884763c2719d456b6', 'Sin tasa al publicar': 'a35c2d9343e1419a914c85fc41675313', 'Sin CBR LAG1': 'ddd9461f58154230a88e05de1bd9c3a1', 'Sin ambas tasas': 'e17d481dee0a44eda77e594a31e37004', 'Sin anuncios secundarios LAG1': '2aedccfc93c8447288a55518925ef1ba', 'Auditoría contemporáneo ausente': 'fd7997671d3b418686f8adc878c5a696'}

def enlace_run(nombre):
    return f"[{nombre}]({TRACKING_URI}/#/experiments/1/runs/{RUNS_REFERENCIA[nombre]})"


## Evidencias históricas — baselines y exclusiones
Se recuperan los resultados originales sin reajustar ni registrar modelos. Los SET de tasas
y anuncios fueron independientes; sus métricas no describen la combinación seleccionada abajo.
La CV de referencia se conserva en los artefactos de los Runs nuevos y no modificó el baseline.


In [11]:
runs_historicos = recuperar_runs(cliente_mlflow,RUNS_REFERENCIA,huella_datos)
with TemporaryDirectory(prefix="ols_historia_cv_") as temporal:
    ruta = cliente_mlflow.download_artifacts(RUNS_REFERENCIA["Sin anuncios secundarios LAG1"],
        "resultados/cv_temporal_todas_configuraciones.json",temporal)
    cv_historica = pd.read_json(ruta)
cv_resumen = cv_historica.groupby("configuracion").agg(
    mae_media=("validation_mae_rub","mean"),mae_sd=("validation_mae_rub","std"))
cv_claves = {"Baseline log":"A","Sin tasa al publicar":"B","Sin CBR LAG1":"C",
             "Sin ambas tasas":"D","Sin anuncios secundarios LAG1":"Anuncios"}
filas_historia = []
mae_referencia = runs_historicos["Baseline log"].data.metrics["validation_mae_rub"]
for nombre in ["Baseline original",*cv_claves]:
    run = runs_historicos[nombre]
    m,p = run.data.metrics,run.data.params
    cv = cv_resumen.loc[cv_claves[nombre]] if nombre in cv_claves else None
    filas_historia.append({"Modelo / Run":enlace_run(nombre),
        "Características":int(p["n_caracteristicas_originales"]),"Columnas OLS":int(p["n_columnas_predictoras"]),
        "MAE Validation RUB":m["validation_mae_rub"],"RMSE Validation RUB":m["validation_rmse_rub"],
        "R² Validation":m["validation_r2"],"MAPE Validation %":m["validation_mape_pct"],
        "CV MAE media ± SD RUB":f"{cv['mae_media']:,.2f} ± {cv['mae_sd']:,.2f}" if cv is not None else "No evaluada",
        "Δ MAE vs log %":100*(m["validation_mae_rub"]/mae_referencia-1)})
historia_modelos = pd.DataFrame(filas_historia)
display(Markdown(tabla_markdown(historia_modelos,decimales=6)))
display(Markdown("**Conclusiones:** log(price) superó al objetivo original. En tasas, D —sin ambas— "
    "mejoró MAE Validation 4,71 % y tuvo menor SD de CV; C obtuvo la menor media de CV. "
    "Quitar el LAG de anuncios aumentó MAE solo 11,05 RUB (0,000556 %), con diferencias temporales mínimas y mixtas; "
    "se conserva como candidata de simplicidad, sin afirmar equivalencia estadística. "
    "La auditoría de la variable contemporánea ausente no fue una ablación: "
    +enlace_run("Auditoría contemporáneo ausente")+"."))


| Modelo / Run | Características | Columnas OLS | MAE Validation RUB | RMSE Validation RUB | R² Validation | MAPE Validation % | CV MAE media ± SD RUB | Δ MAE vs log % |
|---|---|---|---|---|---|---|---|---|
| [Baseline original](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/03bf0a40212846eebf7eb00d68f3c634) | 32 | 185 | 2,582,236.927843 | 3,804,058.411717 | 0.845735 | 21.839803 | No evaluada | 30.046660 |
| [Baseline log](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/ff680a2eb84d4dcb9d1bb64b2ac55036) | 32 | 185 | 1,985,623.415320 | 2,924,096.550772 | 0.908850 | 14.462791 | 1,460,991.39 ± 108,791.74 | 0.000000 |
| [Sin tasa al publicar](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/a35c2d9343e1419a914c85fc41675313) | 31 | 184 | 1,948,242.219768 | 2,878,620.991659 | 0.911663 | 14.168789 | 1,468,248.64 ± 117,921.54 | -1.882592 |
| [Sin CBR LAG1](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/ddd9461f58154230a88e05de1bd9c3a1) | 31 | 184 | 2,006,684.137961 | 2,953,249.397129 | 0.907023 | 14.624214 | 1,432,333.28 ± 98,710.26 | 1.060660 |
| [Sin ambas tasas](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/e17d481dee0a44eda77e594a31e37004) | 30 | 183 | 1,892,014.760854 | 2,803,150.908084 | 0.916234 | 13.732402 | 1,435,853.61 ± 71,743.57 | -4.714321 |
| [Sin anuncios secundarios LAG1](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/2aedccfc93c8447288a55518925ef1ba) | 31 | 184 | 1,985,634.463161 | 2,923,935.745870 | 0.908860 | 14.462804 | 1,460,961.20 ± 108,792.41 | 0.000556 |

**Conclusiones:** log(price) superó al objetivo original. En tasas, D —sin ambas— mejoró MAE Validation 4,71 % y tuvo menor SD de CV; C obtuvo la menor media de CV. Quitar el LAG de anuncios aumentó MAE solo 11,05 RUB (0,000556 %), con diferencias temporales mínimas y mixtas; se conserva como candidata de simplicidad, sin afirmar equivalencia estadística. La auditoría de la variable contemporánea ausente no fue una ablación: [Auditoría contemporáneo ausente](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/fd7997671d3b418686f8adc878c5a696).

## Evidencias históricas — K-Means y Ward
Ambos métodos agruparon efectos de un OLS preliminar con las características originales.
Los mapas de Train completo y de cada fold están preservados como JSON. Las etiquetas de grupos
son internas a cada algoritmo. Se mantienen todos los Runs, métricas y figuras originales.


In [13]:
filas_grupos = []
for k in range(3,11):
    km = runs_historicos[f"KMeans K={k}"].data
    ward = runs_historicos[f"Ward K={k}"].data
    filas_grupos.append({"K":k,"MAE Validation K-Means RUB":km.metrics["validation_mae_rub"],
        "MAE Validation Ward RUB":ward.metrics["validation_mae_rub"],
        "CV MAE K-Means RUB":km.metrics["cv_mae_mean"],"CV MAE Ward RUB":ward.metrics["cv_mae_mean"],
        "Columnas K-Means":int(km.params["n_columnas_predictoras"]),
        "Columnas Ward":int(ward.params["n_columnas_predictoras"]),
        "Runs":enlace_run(f"KMeans K={k}")+" · "+enlace_run(f"Ward K={k}")})
historia_grupos = pd.DataFrame(filas_grupos)
display(Markdown(tabla_markdown(historia_grupos,decimales=2)))
km8 = runs_historicos["KMeans K=8"].data.metrics
km10 = runs_historicos["KMeans K=10"].data.metrics
delta_k8_k10 = 100*(km10["validation_mae_rub"]/km8["validation_mae_rub"]-1)
display(Markdown(f"**Decisión geográfica:** se adopta provisionalmente K-Means K=8 por preferencia "
    f"de compacidad y estabilidad (ARI {km8['cv_ari_mean']:.3f}). K=10 redujo MAE Validation "
    f"otro {abs(delta_k8_k10):.2f} % con dos columnas adicionales; Ward K=10 fue el finalista de Ward. "
    "Esta elección no implica que K=8 tuviera el menor MAE ni que la combinación posterior esté validada."))


| K | MAE Validation K-Means RUB | MAE Validation Ward RUB | CV MAE K-Means RUB | CV MAE Ward RUB | Columnas K-Means | Columnas Ward | Runs |
|---|---|---|---|---|---|---|---|
| 3 | 2,651,725.92 | 2,406,185.50 | 2,027,211.83 | 1,968,476.75 | 59 | 59 | [KMeans K=3](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/38d09624693e4e9eac36c14c60ccba12) · [Ward K=3](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/ebce827914b44a15aead93ed5798234a) |
| 4 | 2,325,779.28 | 2,363,540.81 | 1,852,168.41 | 1,924,773.22 | 60 | 60 | [KMeans K=4](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/33c1f3a196c543ca8776a92db9523827) · [Ward K=4](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/51ef3dcdf4294391bbb0b220f4d2727d) |
| 5 | 2,223,960.04 | 2,251,630.98 | 1,732,959.81 | 1,736,075.02 | 61 | 61 | [KMeans K=5](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/7acb296b04c24338abd5af4bc12c7fa9) · [Ward K=5](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/81e87497a40f491eb537863518009aae) |
| 6 | 2,185,984.12 | 2,207,745.89 | 1,703,882.74 | 1,672,804.59 | 62 | 62 | [KMeans K=6](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/f1429a8e41144005ad0563a74f15d687) · [Ward K=6](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/e9a6012ae7a2451c93e8ef3a2ce01086) |
| 7 | 2,075,714.69 | 2,093,021.69 | 1,604,354.17 | 1,601,549.82 | 63 | 63 | [KMeans K=7](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/ef7ee441b0f941609ed81891eb6a2801) · [Ward K=7](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/85c942d52cd745deaa617273cfecb2ae) |
| 8 | 2,072,596.00 | 2,051,833.45 | 1,560,823.91 | 1,567,573.95 | 64 | 64 | [KMeans K=8](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/ab4ff01bcb544784bf2e33312e128c29) · [Ward K=8](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/3025eb1238a843e3b128a87db8835dfb) |
| 9 | 2,052,344.61 | 2,045,293.38 | 1,554,593.50 | 1,539,066.89 | 65 | 65 | [KMeans K=9](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/86d6fdec3eb349d1a11ed7a930ccb2f8) · [Ward K=9](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/27548a8e778346789a16e9591014c086) |
| 10 | 2,042,408.62 | 2,017,926.30 | 1,525,196.09 | 1,527,623.25 | 66 | 66 | [KMeans K=10](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/29934496f8e04aeda1819706460b82dd) · [Ward K=10](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/4d66a2acee91452884763c2719d456b6) |

**Decisión geográfica:** se adopta provisionalmente K-Means K=8 por preferencia de compacidad y estabilidad (ARI 0.952). K=10 redujo MAE Validation otro 1.46 % con dos columnas adicionales; Ward K=10 fue el finalista de Ward. Esta elección no implica que K=8 tuviera el menor MAE ni que la combinación posterior esté validada.

## Configuración seleccionada para los próximos experimentos

Se utilizará provisionalmente OLS con `log(price_rub)` y Duan, K-Means K=8 y `district_group`
como predictora geográfica. Se excluyen `mortgage_rate_at_listing`, `cbr_key_rate_pct_lag1`
y `n_listings_secondary_lag1`, conservando los otros LAG y `lag1_unavailable`.

**Esta combinación todavía no se evaluó conjuntamente.** El −4,71 % de MAE del experimento
de tasas y el +0,000556 % del experimento individual del LAG justifican decisiones separadas;
no son métricas del DataFrame consolidado. El mapa K=8 conserva el OLS preliminar original:
la exclusión posterior no cambia retrospectivamente las variables que lo originaron.


In [15]:
EXCLUIR_TRABAJO = ["mortgage_rate_at_listing","cbr_key_rate_pct_lag1","n_listings_secondary_lag1"]
RUN_KMEANS8 = RUNS_REFERENCIA["KMeans K=8"]
configuracion_seleccionada = {
    "version":1,"modelo":"OLS","target_original":"price_rub","target_entrenamiento":"log(price_rub)",
    "retransformacion":"exp(pred_log) * Duan estimado en Train",
    "agrupamiento":{"algoritmo":"KMeans","K":8,"run_id":RUN_KMEANS8,
        "artefacto":ARTEFACTO_MAPAS,"contexto":"train_completo",
        "predictoras_ols_preliminar":list(predictoras_base)},
    "excluir":list(EXCLUIR_TRABAJO),"dataset_sha256":huella_datos,
    "conservar":["n_listings_newbuild_lag1","n_listings_rental_lag1","lag1_unavailable"],
    "evaluada_conjuntamente":False,
    "fundamento_experimentos_independientes":{
        "tasas_run":RUNS_REFERENCIA["Sin ambas tasas"],"anuncios_lag1_run":RUNS_REFERENCIA["Sin anuncios secundarios LAG1"],
        "nota":"No atribuir a esta combinación las métricas de los experimentos individuales."},
}


## DataFrame de trabajo — Configuración seleccionada

Se recupera el mapa K-Means K=8 aprendido solo con Train y se aplica sin clustering.
`df_trabajo` es una copia independiente: pierde físicamente las tres columnas descartadas,
conserva precio y particiones originales y mantiene `district` únicamente como auxiliar.
`district_group` es categórica, con las ocho categorías definidas por el mapa de Train.
La asignación a las filas reservadas de Test es solo preparación estructural: no se ajusta
ningún transformador ni se predicen o evalúan sus precios. No se aplica One-Hot global.


In [17]:
contexto_mapa_completo = contexto_entrenamiento(conjuntos["train"],dataset_sha256=huella_datos,
    caracteristicas=predictoras_base,numericas=columnas_numericas,categoricas=columnas_ohe,
    preprocesamiento=config_preprocesamiento)
agrupamiento_k8 = recuperar_agrupamiento(cliente_mlflow,algoritmo="KMeans",k=8,run_id=RUN_KMEANS8,
    contexto_esperado=contexto_mapa_completo,cache=cache_transformaciones)
df_trabajo,predictoras_trabajo = preparar_df_seleccionado(datos_base,agrupamiento_k8,EXCLUIR_TRABAJO,
    predictoras_base=predictoras_base,uso="preparacion_reservada")
categoricas_trabajo = ["district_group" if c=="district" else c for c in columnas_ohe]
numericas_trabajo = [c for c in columnas_numericas if c not in EXCLUIR_TRABAJO]
auxiliares_trabajo = [c for c in df_trabajo if c not in predictoras_trabajo and c!=objetivo]
df_trabajo_train = df_trabajo.loc[df_trabajo["dataset_split"].eq("train")].copy()
df_trabajo_validation = df_trabajo.loc[df_trabajo["dataset_split"].eq("validation")].copy()
assert len(df_trabajo)==len(datos_base) and df_trabajo.index.equals(datos_base.index)
assert not set(EXCLUIR_TRABAJO).intersection(df_trabajo.columns)
assert set(configuracion_seleccionada["conservar"]) <= set(predictoras_trabajo)
assert "district" not in predictoras_trabajo and "district" in auxiliares_trabajo
assert "district_group" in categoricas_trabajo
assert df_trabajo_train["district_group"].nunique()==8 and not df_trabajo["district_group"].isna().any()
assert not datos_base.loc[datos_base["dataset_split"].eq("train"),"district"].map(agrupamiento_k8["mapa"]).isna().any()
assert set(predictoras_trabajo)==set(numericas_trabajo+categoricas_trabajo)
pd.testing.assert_series_equal(df_trabajo[objetivo],datos_base[objetivo])
assert huella_tabla(datos_base)==huella_base_memoria


## Uso posterior y validación temporal sin leakage

Para el ajuste completo y Validation externa se utilizarán `df_trabajo_train`,
`df_trabajo_validation` y `predictoras_trabajo`. Dentro de CV se parte de `datos_base`
y se llama a `preparar_fold_trabajo(fold)`: recupera el mapa exclusivo de ese entrenamiento
y aplica después las mismas exclusiones. No se reutiliza el mapa de Train completo en CV.

Se conserva expresamente la definición original de los efectos de distrito. Si se cambia
ese OLS preliminar o el preprocesamiento, la recuperación rechaza el contexto incompatible;
recalcular agrupamientos requerirá otro experimento. No se ejecuta ahora Forward, Backward,
Best Subset, Ridge, LASSO ni entrenamiento del modelo combinado.


In [19]:
def preparar_fold_trabajo(fold):
    return preparar_fold_seleccionado(datos_base,fold,cliente=cliente_mlflow,run_id=RUN_KMEANS8,
        excluir=EXCLUIR_TRABAJO,dataset_sha256=huella_datos,predictoras_base=predictoras_base,
        numericas=columnas_numericas,categoricas=columnas_ohe,preprocesamiento=config_preprocesamiento,
        n_folds=N_FOLDS,meses_validacion=MESES_VALID_FOLD,cache=cache_transformaciones)

# En un futuro experimento: preparar_fold_trabajo(1), luego ajustar solo con su Train.
# crear_preprocesador(numericas=numericas_trabajo,categoricas=categoricas_trabajo)
# Las columnas OLS se medirán tras ajustar ese preprocesador; todavía no se han calculado.


## Trazabilidad de la configuración provisional
Se guarda una nota Markdown y su configuración JSON bajo una ruta nueva del Run K=8.
No se crea un Run, no se registran métricas del modelo combinado y no se sobrescriben
notas ni artefactos de experimentos previos. Una configuración distinta deberá versionarse.


In [21]:
configuracion_seleccionada["predictoras_trabajo"] = list(predictoras_trabajo)
configuracion_seleccionada["contexto_mapa_sha256"] = huella_json(contexto_mapa_completo)
configuracion_seleccionada["id_transformacion"] = agrupamiento_k8["id_transformacion"]
configuracion_seleccionada["ids_folds"] = [f"{RUN_KMEANS8}:fold_{f}" for f in range(1,N_FOLDS+1)]
configuracion_seleccionada["dimensiones_preparadas"] = {
    "filas":len(df_trabajo),"columnas":len(df_trabajo.columns),"predictoras":len(predictoras_trabajo)}
configuracion_md = ("# Configuración provisional para Feature Selection\n\n"
    +f"OLS log(price_rub), Duan, K-Means K=8. Run de origen: {RUN_KMEANS8}.\n\n"
    +"Exclusiones: "+", ".join(EXCLUIR_TRABAJO)+".\n\n"
    +"Decisiones independientes: quitar ambas tasas redujo MAE Validation 4,71 %; quitar el LAG de "
     "anuncios lo incrementó 0,000556 %. K=8 es la representación geográfica provisional elegida.\n\n"
    +"**La combinación no se evaluó conjuntamente y no tiene métricas predictivas propias.**\n\n"
    +"El mapa se aprendió solo con Train, usando las características originales del baseline. "
     "En CV se deben recuperar los mapas de cada fold; las exclusiones se aplican después. "
     "No se exportan propiedades ni se ajustan transformadores sobre Test.")
carpeta_config = "configuraciones_provisionales"
ruta_config = carpeta_config+"/kmeans8_sin_tasas_sin_secondary_lag1_v1.json"
ruta_nota = carpeta_config+"/kmeans8_sin_tasas_sin_secondary_lag1_v1.md"
antes_registro = cliente_mlflow.get_run(RUN_KMEANS8)
archivos = {a.path for a in cliente_mlflow.list_artifacts(RUN_KMEANS8,carpeta_config)}
if ruta_config in archivos:
    assert leer_json_remoto(cliente_mlflow,RUN_KMEANS8,ruta_config)==configuracion_seleccionada, "Versionar una configuración diferente."
else:
    cliente_mlflow.log_dict(RUN_KMEANS8,configuracion_seleccionada,ruta_config)
if ruta_nota not in archivos:
    with TemporaryDirectory(prefix="configuracion_ols_") as temporal:
        nota = Path(temporal)/Path(ruta_nota).name
        nota.write_text(configuracion_md,encoding="utf-8")
        cliente_mlflow.log_artifact(RUN_KMEANS8,str(nota),artifact_path=carpeta_config)
assert leer_json_remoto(cliente_mlflow,RUN_KMEANS8,ruta_config)==configuracion_seleccionada
despues_registro = cliente_mlflow.get_run(RUN_KMEANS8)
assert despues_registro.data.metrics==antes_registro.data.metrics
assert despues_registro.data.params==antes_registro.data.params and despues_registro.data.tags==antes_registro.data.tags
assert hashlib.sha256(archivo_datos.read_bytes()).hexdigest()==huella_datos
assert huella_tabla(datos_base)==huella_base_memoria
display(Markdown("Configuración provisional guardada y verificada en "+enlace_run("KMeans K=8")+". Sin nuevos Runs ni métricas."))


Configuración provisional guardada y verificada en [KMeans K=8](https://dagshub.com/matiasgastonlonghino/MOSCOW_IDA.mlflow/#/experiments/1/runs/ab4ff01bcb544784bf2e33312e128c29). Sin nuevos Runs ni métricas.

## DataFrame que utilizaremos a partir de ahora

`df_trabajo` está disponible en memoria. La vista siguiente se genera desde sus valores reales:
primeras diez filas, dimensiones, esquema completo, predictoras, exclusiones y particiones.
La vista previa permite desplazamiento horizontal para consultar todas las columnas.


In [23]:
display(Markdown(f"**df_trabajo: {len(df_trabajo):,} filas × {df_trabajo.shape[1]} columnas; "
    f"{len(predictoras_trabajo)} predictoras antes de One-Hot.** "
    "El número de columnas OLS aún no se ha medido. El precio original sigue en RUB."))
display(HTML('<div style="overflow-x:auto;max-width:100%">'
    +df_trabajo.head(10).to_html(index=False,float_format=lambda v:f"{v:,.3f}")+'</div>'))
esquema_trabajo = pd.DataFrame([{"Columna":c,"Tipo":str(df_trabajo[c].dtype),
    "Rol":"Objetivo original RUB" if c==objetivo else "Predictora" if c in predictoras_trabajo else "Auxiliar",
    "Tratamiento":"Categórica nominal (8 grupos)" if c=="district_group" else
        "Categórica; OHE futuro" if c in categoricas_trabajo else "Numérica" if c in numericas_trabajo else "Conservar"}
    for c in df_trabajo.columns])
display(Markdown(tabla_markdown(esquema_trabajo)))
comprobacion_exclusiones = pd.DataFrame({"Variable excluida":EXCLUIR_TRABAJO,
    "Ausente de df_trabajo":["Sí" if c not in df_trabajo else "No" for c in EXCLUIR_TRABAJO]})
display(Markdown(tabla_markdown(comprobacion_exclusiones)))
particiones_trabajo = df_trabajo.groupby("dataset_split").agg(
    registros=("id","size"),desde=("date_posted_dt","min"),hasta=("date_posted_dt","max")).reindex(orden_splits)
particiones_trabajo["porcentaje_real"] = particiones_trabajo["registros"]/len(df_trabajo)*100
display(Markdown(tabla_markdown(particiones_trabajo,indice=True,decimales=2)))
decisiones_trabajo = pd.DataFrame({"Componente":["Target","Agrupamiento","Variable geográfica predictora",
    "Tasa al publicar","Tasa CBR LAG1","Anuncios secundarios LAG1","Otros indicadores LAG","Particiones","Estado conjunto"],
    "Configuración":["log(price_rub) solo al entrenar; price_rub permanece en RUB","K-Means, K=8", "district_group",
        *("Excluida" if c not in df_trabajo else "Presente" for c in EXCLUIR_TRABAJO),
        ", ".join(c for c in configuracion_seleccionada["conservar"] if c in df_trabajo),
        "Persistidas: Train 70 %, Validation 10 %, Test 20 % nominales; efectivos en tabla", "No evaluado"]})
display(Markdown(tabla_markdown(decisiones_trabajo)))
assert huella_tabla(datos_base)==huella_base_memoria
display(Markdown("**Disponible para los próximos experimentos.** Sin OHE global, nuevos entrenamientos ni evaluación de Test."))


**df_trabajo: 49,753 filas × 36 columnas; 29 predictoras antes de One-Hot.** El número de columnas OLS aún no se ha medido. El precio original sigue en RUB.

id,price_rub,lat,lon,total_area,living_area,kitchen_area,rooms,floor,total_floors,building_year,ceiling_height,metro_distance_min,to_center_km,metro_lat,metro_lon,metro_year_opened,metro_to_center_km,has_balcony,metro_distance_type,seller_type,building_year_inconsistent,months_since_start,n_listings_newbuild_lag1,n_listings_rental_lag1,lag1_unavailable,district,building_type,renovation,metro_line,month_of_year,okrug,date_posted_dt,year_month,dataset_split,district_group
S000598,11678000,55.881,37.616,99.600,67.900,15.500,4,3,17,"1,971.000",2.500,34,14.160,55.854,37.653,1978,11.340,0,1,1,0,0,NaN,NaN,1,Yuzhnoye Medvedkovo,khrushchev,designer,Kaluzhsko-Rizhskaya,1,SVAO,2020-01-01,2020-01,train,G04
S001135,13501000,55.717,37.651,76.100,45.600,13.900,2,5,9,"2,011.000",2.700,15,4.560,55.710,37.623,1983,4.860,0,0,1,0,0,NaN,NaN,1,Danilovsky,modern_panel,cosmetic,Serpukhovsko-Timiryazevskaya,1,YuAO,2020-01-01,2020-01,train,G05
S005517,6733000,55.627,37.628,87.100,54.800,7.100,3,9,17,"2,004.000",2.700,32,14.160,55.655,37.603,1983,11.020,0,1,1,0,0,NaN,NaN,1,Chertanovo Tsentralnoye,modern_panel,no_renovation,Serpukhovsko-Timiryazevskaya,1,YuAO,2020-01-01,2020-01,train,G03
S006770,1927000,55.396,37.510,47.600,31.800,13.300,1,10,17,NaN,2.700,189,40.360,55.759,37.752,1979,8.200,0,1,1,1,0,NaN,NaN,1,Shchapovskoye,monolith,cosmetic,Kalininskaya,1,TAO,2020-01-01,2020-01,train,G01
S011704,7701000,55.797,37.630,52.200,30.700,8.300,1,11,25,"1,971.000",2.800,31,4.770,55.757,37.660,1995,2.450,1,1,0,0,0,NaN,NaN,1,Alekseevsky,brick,cosmetic,Lyublinsko-Dmitrovskaya,1,SVAO,2020-01-01,2020-01,train,G05
S014492,3514000,55.703,37.851,41.800,24.700,10.100,1,21,22,"1,984.000",2.800,40,15.460,55.759,37.752,1979,8.200,1,1,0,0,0,NaN,NaN,1,Kosino-Ukhtomsky,brick,cosmetic,Kalininskaya,1,VAO,2020-01-01,2020-01,train,G03
S015859,11669000,55.820,37.741,59.300,34.800,15.800,2,2,17,"2,017.000",2.700,3,10.490,55.796,37.716,1965,7.560,1,0,1,0,0,NaN,NaN,1,Preobrazhenskoye,monolith,no_renovation,Sokolnicheskaya,1,VAO,2020-01-01,2020-01,train,G05
S019272,5746000,55.936,37.519,79.900,48.400,13.300,3,25,25,"1,963.000",2.700,61,21.170,55.819,37.498,1964,10.560,1,1,0,0,0,NaN,NaN,1,Severny,khrushchev,cosmetic,Zamoskvoretskaya,1,SVAO,2020-01-01,2020-01,train,G03
S019502,8155000,55.629,37.685,53.600,33.800,9.300,1,3,17,"2,001.000",3.000,33,14.410,55.710,37.623,1983,4.860,1,1,1,0,0,NaN,NaN,1,Moskvorechye-Saburovo,monolith,euro,Serpukhovsko-Timiryazevskaya,1,YuAO,2020-01-01,2020-01,train,G04
S021402,12641000,55.773,37.487,60.800,39.800,10.300,2,7,17,"2,002.000",2.700,40,8.650,55.770,37.596,1938,2.360,1,1,1,0,0,NaN,NaN,1,Khoroshyovo-Mnyovniki,brick,euro,Zamoskvoretskaya,1,SZAO,2020-01-01,2020-01,train,G05


| Columna | Tipo | Rol | Tratamiento |
|---|---|---|---|
| id | str | Auxiliar | Conservar |
| price_rub | int64 | Objetivo original RUB | Conservar |
| lat | float64 | Predictora | Numérica |
| lon | float64 | Predictora | Numérica |
| total_area | float64 | Predictora | Numérica |
| living_area | float64 | Predictora | Numérica |
| kitchen_area | float64 | Predictora | Numérica |
| rooms | int64 | Predictora | Numérica |
| floor | int64 | Predictora | Numérica |
| total_floors | int64 | Predictora | Numérica |
| building_year | float64 | Predictora | Numérica |
| ceiling_height | float64 | Predictora | Numérica |
| metro_distance_min | int64 | Predictora | Numérica |
| to_center_km | float64 | Predictora | Numérica |
| metro_lat | float64 | Predictora | Numérica |
| metro_lon | float64 | Predictora | Numérica |
| metro_year_opened | int64 | Predictora | Numérica |
| metro_to_center_km | float64 | Predictora | Numérica |
| has_balcony | int64 | Predictora | Numérica |
| metro_distance_type | int64 | Predictora | Numérica |
| seller_type | int64 | Predictora | Numérica |
| building_year_inconsistent | int64 | Predictora | Numérica |
| months_since_start | int64 | Predictora | Numérica |
| n_listings_newbuild_lag1 | float64 | Predictora | Numérica |
| n_listings_rental_lag1 | float64 | Predictora | Numérica |
| lag1_unavailable | int64 | Predictora | Numérica |
| district | str | Auxiliar | Conservar |
| building_type | str | Predictora | Categórica; OHE futuro |
| renovation | str | Predictora | Categórica; OHE futuro |
| metro_line | str | Predictora | Categórica; OHE futuro |
| month_of_year | int64 | Predictora | Categórica; OHE futuro |
| okrug | str | Auxiliar | Conservar |
| date_posted_dt | datetime64[us] | Auxiliar | Conservar |
| year_month | str | Auxiliar | Conservar |
| dataset_split | str | Auxiliar | Conservar |
| district_group | category | Predictora | Categórica nominal (8 grupos) |

| Variable excluida | Ausente de df_trabajo |
|---|---|
| mortgage_rate_at_listing | Sí |
| cbr_key_rate_pct_lag1 | Sí |
| n_listings_secondary_lag1 | Sí |

| dataset_split | registros | desde | hasta | porcentaje_real |
|---|---|---|---|---|
| train | 34853 | 2020-01-01 00:00:00 | 2024-05-31 00:00:00 | 70.05 |
| validation | 5258 | 2024-06-01 00:00:00 | 2025-01-31 00:00:00 | 10.57 |
| test | 9642 | 2025-02-01 00:00:00 | 2026-04-29 00:00:00 | 19.38 |

| Componente | Configuración |
|---|---|
| Target | log(price_rub) solo al entrenar; price_rub permanece en RUB |
| Agrupamiento | K-Means, K=8 |
| Variable geográfica predictora | district_group |
| Tasa al publicar | Excluida |
| Tasa CBR LAG1 | Excluida |
| Anuncios secundarios LAG1 | Excluida |
| Otros indicadores LAG | n_listings_newbuild_lag1, n_listings_rental_lag1, lag1_unavailable |
| Particiones | Persistidas: Train 70 %, Validation 10 %, Test 20 % nominales; efectivos en tabla |
| Estado conjunto | No evaluado |

**Disponible para los próximos experimentos.** Sin OHE global, nuevos entrenamientos ni evaluación de Test.